# Coupled Mamba: adapted reference and controls

This Stage 3 notebook establishes the local Coupled Mamba adaptation and its controls before later task-specific experiments. Its three streams share a codon grid; applying the method to separate RNA–protein or protein–ligand partners requires a new input and evaluation design. The documented departures from the paper remain part of the reference.

**The question.** Does adding recurrent state to fusion help beyond the BioLangFusion heads? Concatenation combines modalities at each codon, the gated head assigns modality weights from sequence summaries, and cross-attention lets each position attend across the three streams. Cross-attention can therefore connect different positions; its distinction from recurrence is that it does not carry an explicit state along the codon axis. The frozen encoders can already supply contextual token representations in every comparison.

Coupled Mamba adds that recurrent state (Li et al., NeurIPS 2024). Each modality runs its own selective state-space block, and at every codon each modality's transition is applied to the pooled state of all three. Information accumulates along the codon axis while the modalities stay coupled. The implementation and the project's four deviations from the paper are documented in [`shared_code/fusion/coupled_mamba.py`](../shared_code/fusion/coupled_mamba.py).

**What this notebook tests.** With the encoders and evaluation held fixed, compare coupled recurrence with independent recurrent streams, a projection bottleneck, and the saved concatenation and cross-attention heads. The predictions below propose where recurrence and coupling might help. These comparisons test those architectural choices under this setup; their scores alone cannot identify the biological mechanism or establish that useful information is exclusively local or long-range.

**Three designs, so a result can be attributed:**

| Design | What it is | Isolates |
| --- | --- | --- |
| `coupled_mamba` | Projections to width 128, two coupled Mamba layers, fusion to 600 | |
| `uncoupled_mamba` | The same layers and parameter count, each modality's state on its own chain | the coupling, against `coupled_mamba` |
| `mamba_bottleneck` | The projections to width 128 and the final fusion, no Mamba layers | the recurrence, against `uncoupled_mamba` |

**Matched to the references.** Every fit uses the Phase A recipe exactly: BioLangFusion's optimizer and stopping settings, 100 TextCNN channels, the Equation 1 grid, seed 0, the published splits and BioLangFusion's own encoders (NT v2 100M, RNA-FM, ESM-2 8M). Only the fusion head differs. The references remain the saved seed-0 fits in `Code/stage4/results/phaseA/` (and `Code/stage3/results/fungal_expression/` for fungal). New fits require recorded input identity. The named historical references can instead use the explicitly qualified comparison evidence described below; this does not recover their original training provenance. Paired contrasts still require matching ordered-input, target and partition fingerprints and valid saved predictions.

**Default Run All reads saved results** and reports missing files. Training starts only when `READ_SAVED_RESULTS = False`. Statistical analysis is separate: `RUN_BOOTSTRAP = True` explicitly recalculates the paired intervals from saved predictions, without saving a report. No standalone Mamba bootstrap report has been retained, so default reading reports that limitation; historical notebook outputs keep their original execution provenance.

---

## Setup

Use the [repository environment](../../README.md#notebook-dependencies). Execution switches, historical reference locations and the prediction-head recipe remain visible below; reading does not request new fits.

### Imports


In [16]:
# Environment, Python module paths, elapsed time and filesystem paths.
import os
import sys
import time
from pathlib import Path
from types import SimpleNamespace

# Set before importing torch: unsupported Apple/MPS operations may fall back to CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables.
import numpy as np
import pandas as pd

# Model computation and hardware availability.
import torch

# Notebook table display.
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

### Hardware and display

These diagnostics describe the current kernel, not the machine that produced a saved result. The configured device is used only when fitting is enabled; reading scores does not itself train on that device.


In [17]:
# The experiment's device preference, as in Phase A: Apple/MPS, NVIDIA/CUDA, then CPU.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(
    "PyTorch:", torch.__version__,
    "| CUDA available:", torch.cuda.is_available(),
    "| MPS available:", torch.backends.mps.is_available(),
)
print("Configured compute device:", DEVICE)

PyTorch: 2.12.1 | CUDA available: False | MPS available: True
Configured compute device: mps


### Project paths and shared code

Locate `Code/shared_code` without changing the working directory. Importing `coupled_mamba` also registers its fusion heads; the list below confirms the available names.


In [18]:
# Find Code/shared_code from the repository root or from inside Code.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )
# Make shared_code importable without changing the kernel working directory.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import (blf, blf_cache_recovery, coupled_mamba, heads,
                                run_identity, swap_results, train)

print("Registered fusion heads:", sorted(heads.FUSIONS))

Registered fusion heads: ['concat', 'coupled_mamba', 'cross', 'mamba_bottleneck', 'mil', 'uncoupled_mamba']


### Execution settings and file locations

`READ_SAVED_RESULTS` controls fitting; `RUN_BOOTSTRAP` independently requests statistics from saved predictions. The historical reference exception remains limited to its named evidence files. Keep the result and reference destinations distinct.


In [19]:
# Default Run All reads saved scores without training or statistical resampling.
READ_SAVED_RESULTS = False
RUN_BOOTSTRAP = False  # Explicit analysis of saved predictions; independent of training.

# Comparison-only exception for the 24 reviewed reference files; disable for strict identity only.
ALLOW_LEGACY_REFERENCE_REUSE = True
LEGACY_REFERENCE_RECORD = CODE_DIR / "stage4" / "legacy_reference_reuse.json"

# All four benchmarks with saved references. To start with one, shorten this tuple.
BENCHMARKS = ("ecoli_proteins", "cov_vac_degradation", "fungal_expression")

# Cheapest first: the bottleneck has no Mamba layers; the uncoupled design runs three scans.
DESIGNS = ("mamba_bottleneck", "coupled_mamba", "uncoupled_mamba")

# Inputs and references are read from Stages 3 and 4; this experiment writes only here.
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "tier4_designs"
PHASE_A = CODE_DIR / "stage4" / "results" / "phaseA"
# Reference locations. Fungal's native references come from Stage 3; grid-RNA comes from Phase A.
REFERENCE_DIRS = {
    "cov_vac_degradation": (PHASE_A / "cov_vac_degradation",),
    "ecoli_proteins": (PHASE_A / "ecoli_proteins",),
    "mrna_stability": (PHASE_A / "mrna_stability",),
    "fungal_expression": (CODE_DIR / "stage3" / "results" / "fungal_expression",
                          PHASE_A / "fungal_expression"),
}



### Qualified provenance for the October token caches

A new fit records the identity of everything it reads, including a SHA-256 for every token-cache
shard. The caches extracted in October predate those fields, so strict validation refuses them
before training starts. The cell below writes one reviewed recovery record per benchmark: it
hashes every shard, recomputes the ordered-input fingerprint from the benchmark table, and checks
that fingerprint against a completed Newton fit, whose saved identity carries the fingerprint
recorded at extraction time on another machine.

What this establishes is that these caches hold the inputs they claim and have not changed since
the inspection. What it cannot establish is which program produced the stored vectors; runs that
read a recovered cache carry that reconstruction label in their saved identity, and re-extraction
with the current writer remains the route where the producer matters.


In [20]:
# One record per benchmark, written once and reused; reading an existing record changes nothing.
RECOVERY_DIR = CODE_DIR / "stage4" / "results" / "cache_recovery"
# Completed Newton fits, whose identity carries extraction-time cache provenance.
NEWTON_PHASE_A = CODE_DIR / "stage4" / "results" / "newton_h100_seeds012" / "outputs" / "phaseA"

RECOVERY_RECORDS = {}
for benchmark in BENCHMARKS:
    path = RECOVERY_DIR / f"{benchmark}-cache-recovery.json"
    RECOVERY_RECORDS[benchmark] = path
    if path.exists():
        print(f"{benchmark}: record already written, {path.name}")
        continue
    # One fit is enough: every fit of a benchmark records the same caches.
    corroborating = sorted((NEWTON_PHASE_A / benchmark).glob("*__concat__*seed0.json"))[:1]
    streams = blf.BenchmarkStreams.open(benchmark, CACHE_ROOT)
    try:
        record = blf_cache_recovery.write_recovery_record(
            streams, path, corroborating_results=corroborating)
    finally:
        # This cell owns the opened caches whether or not the inspection succeeds.
        streams.close()
    print(f"{benchmark}: {record['record_id']}, "
          f"corroborated by Newton for {record['corroborated_modalities']}")


ecoli_proteins: record already written, ecoli_proteins-cache-recovery.json
cov_vac_degradation: record already written, cov_vac_degradation-cache-recovery.json
fungal_expression: record already written, fungal_expression-cache-recovery.json


### Experiment settings

The single-encoder candidates and predictor configuration retain the Phase A comparison. The configuration's seed is a training seed; it does not choose bootstrap draws.


In [21]:
# Phase A's four eligible single-encoder baselines; the best is chosen on validation.
SINGLE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot", "unimodal-grid:rna")

# Phase A's recipe exactly: the trainer's BioLangFusion defaults, 100 channels, Equation 1, seed 0.
CONFIG = train.TrainConfig(channels=100, tconv="equation", seed=0)
print(CONFIG)

TrainConfig(lr=3e-05, weight_decay=1e-05, batch_size=32, max_epochs=500, early_stopping_patience=20, plateau_patience=5, entropy_lambda=0.0, channels=100, tconv='equation', seed=0)


## Predictions, written before any fit

Write down what each design should do on each benchmark, with Chase, **before** the first fit, and do not edit this cell afterward. A prediction written after the result is not a test of the interpretation.

The proposal below follows the project's reading of each benchmark. Replace it with your own if you disagree.

| Benchmark | What drives the target, by our reading | Prediction |
| --- | --- | --- |
| E. coli expression | Codon usage, close to a bag of codons | Local. `coupled_mamba` ties `concat`; coupling adds nothing over `uncoupled_mamba` |
| Fungal expression | Composition (protein and codon), per the swap results | Local. Same as E. coli |
| mRNA stability | Codon optimality, also largely compositional | Local. Same as E. coli |
| CoV-Vac degradation | Chemical degradation of short constructs, set by which bases are paired, a long-range property | Long-range. `coupled_mamba` above `concat` and above `uncoupled_mamba` |

Improvement on the predicted benchmarks would support this hypothesis under the selected setup. Broader gains or no gains would prompt revisiting it. Neither outcome alone proves local or long-range biological dependence: the encoders already carry context, and the compared heads also differ in capacity and optimization. Treat the proposed explanations above as hypotheses to test, not conclusions supplied by a score difference.

## Train the designs

Skipped unless `READ_SAVED_RESULTS = False`. The trainer checks a completed fit before reusing it; an incompatible result raises an error. New fits use the existing input/cache identity writer. Legacy completed design files remain unqualified; the reference-only reuse option below cannot authorize a fit, checkpoint resume or cache reuse. Runtime for these designs has not been established here, so no duration relative to Phase A is assumed.


In [22]:
if READ_SAVED_RESULTS:
    print("READ_SAVED_RESULTS is True: no training. Set it to False to fill missing fits.")
else:
    for benchmark in BENCHMARKS:
        started = time.time()
        streams = blf.BenchmarkStreams.open(benchmark, CACHE_ROOT)
        try:
            for design in DESIGNS:
                destination = RESULTS_ROOT / benchmark
                records = blf_cache_recovery.execution_records(
                    streams, design, CONFIG, destination,
                    repository=CODE_DIR.parent, device=DEVICE,
                    recovery=RECOVERY_RECORDS[benchmark],
                )
                train.run(streams, design, CONFIG, DEVICE, destination, **records)
        except BaseException as error:
            # A stream-close failure must not hide the fit failure or interruption.
            try:
                streams.close()
            except BaseException as cleanup_error:
                if hasattr(error, "add_note"):
                    error.add_note(f"Stream cleanup also failed: {cleanup_error}")
            raise
        else:
            streams.close()
        print(f"{benchmark}: {(time.time() - started) / 60:.1f} min")


/Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/shared_code/fusion/blf_cache_recovery.py:288: UserWarning: Using a reviewed historical reconstruction for legacy BioLangFusion caches; their current bytes are verified, while extraction-time input proof and producer details remain unknown.
  caches.update(recovered_cache_records(streams, recovery, verification=verification,


  ecoli_proteins__mamba_bottleneck__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 1.1145  val 0.4313  best 0.4313 @ 0
    epoch  10  train loss 0.8513  val 0.5577  best 0.5742 @ 5
    epoch  20  train loss 0.7633  val 0.5838  best 0.5934 @ 16
    epoch  30  train loss 0.7454  val 0.5687  best 0.5934 @ 16
  ecoli_proteins__mamba_bottleneck__lambda0.0__ch100__equation__seed0: test 0.5744 (val 0.5934 at epoch 16, 6.4 min)


/Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/shared_code/fusion/blf_cache_recovery.py:288: UserWarning: Using a reviewed historical reconstruction for legacy BioLangFusion caches; their current bytes are verified, while extraction-time input proof and producer details remain unknown.
  caches.update(recovered_cache_records(streams, recovery, verification=verification,


  ecoli_proteins__coupled_mamba__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 1.1150  val 0.4162  best 0.4162 @ 0
    epoch  10  train loss 0.8287  val 0.5412  best 0.5783 @ 5
    epoch  20  train loss 0.7258  val 0.5687  best 0.5783 @ 5
  ecoli_proteins__coupled_mamba__lambda0.0__ch100__equation__seed0: test 0.5604 (val 0.5783 at epoch 5, 52.7 min)


/Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/shared_code/fusion/blf_cache_recovery.py:288: UserWarning: Using a reviewed historical reconstruction for legacy BioLangFusion caches; their current bytes are verified, while extraction-time input proof and producer details remain unknown.
  caches.update(recovered_cache_records(streams, recovery, verification=verification,


  ecoli_proteins__uncoupled_mamba__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 1.1150  val 0.4093  best 0.4093 @ 0
    epoch  10  train loss 0.8273  val 0.5316  best 0.5714 @ 5
    epoch  20  train loss 0.7199  val 0.5714  best 0.5755 @ 14
    epoch  30  train loss 0.7114  val 0.5646  best 0.5783 @ 21


KeyboardInterrupt: 

## Read the designs and the references

Scores are Spearman correlation for the regression benchmarks and accuracy (percent) for E. coli. The single-encoder candidates are Phase A's three native-resolution encoders and the RNA stream after the codon grid. Selection requires all four compatible, unambiguous results and matching comparison cohorts; validation chooses the best, with the declared candidate order breaking ties. Missing, invalid or incompatible evidence leaves “best single” unavailable rather than selecting from a smaller set.

**Qualified historical reuse.** `ALLOW_LEGACY_REFERENCE_REUSE` permits only the 24 seed-0 files named in the [reuse record](../stage4/legacy_reference_reuse.json), at their recorded paths and exact SHA-256 hashes. Their input signatures come from the existing later verification reports, with the original encoder mapping accepted from the historical notebook and run records. We accept those cohorts for these comparisons because the dataset and row-selection history agree and the available saved targets match. This is an explicit reconstruction assumption: the exact producing code, environment and cache-to-fit binding remain unestablished. It does not add `identity` or `producer` fields to the old results. See the [decision and evidence](../../docs/run-records.md#2026-11-12-scoped-legacy-reference-reuse).

The coverage and contrast tables label **qualified legacy reference reuse** wherever it is used. Set the option to `False` to require original recorded identity throughout. Changed or unlisted references and malformed modern identities remain ineligible; new design fits cannot use this exception. Individual scores remain readable when comparison evidence is unavailable.

The historical fungal RNA-FM and ESM-2 8M files lack per-row predictions. Their validation scores can participate in complete baseline selection, but any paired contrast consuming either still remains unavailable. No prediction array is inferred or regenerated.


In [ ]:
def comparison_inputs(record, reference_sources=None):
    """Keep recorded identity separate from explicitly accepted historical evidence."""
    if "identity" in record:
        # Invalid modern metadata must fail rather than fall back to legacy reuse.
        return run_identity.recorded_blf_inputs(record)
    if not ALLOW_LEGACY_REFERENCE_REUSE:
        raise ValueError("Legacy reference reuse is disabled; recorded input identity is required.")
    source = (reference_sources or {}).get(record["variant"])
    if source is None:
        raise ValueError("No approved reference source for this legacy result.")
    return run_identity.legacy_reference_inputs(
        record, path=source, repository=CODE_DIR.parent,
        recovery_path=LEGACY_REFERENCE_RECORD,
    )


def comparison_provenance(*records):
    """Label successful checks without promoting legacy evidence to producer identity."""
    if any("identity" not in record for record in records):
        return "qualified legacy reference reuse"
    return "recorded inputs"


In [ ]:
def read_design_results(benchmark):
    """Read compatible recipes and retain explanations for excluded records."""
    found, issues, reference_sources = {}, [], {}
    sources = (
        (REFERENCE_DIRS[benchmark], ("concat", "cross", *SINGLE_VARIANTS), True),
        ((RESULTS_ROOT / benchmark,), DESIGNS, False),
    )
    for folders, variants, native_fallback in sources:
        report = swap_results.read_results(
            folders, benchmark=benchmark, config=vars(CONFIG),
            encoders=blf.ENCODER_OF_MODALITY, native_fallback=native_fallback,
            variants=variants,
        )
        # Ambiguous identities are already withheld by the shared reader.
        found.update({entry["data"]["variant"]: entry["data"] for entry in report.records})
        if native_fallback:
            # Only reference-folder records may request the scoped legacy exception.
            reference_sources.update({
                entry["data"]["variant"]: entry["path"] for entry in report.records
            })
        issues.extend(
            issue for issue in report.issues
            if issue.get("variant") is None or issue["variant"] in variants
        )
        issues.extend({"source": "", "reason": problem} for problem in report.duplicates)
    return found, issues, reference_sources


In [ ]:
def validation_selected_single(found, reference_sources=None):
    """Choose among the complete declared set on matching comparison inputs."""
    missing = [variant for variant in SINGLE_VARIANTS if variant not in found]
    if missing:
        raise ValueError("Single-encoder coverage incomplete: " + ", ".join(missing))

    # Apply the declared evidence policy before comparing validation scores.
    reference_inputs = comparison_inputs(found[SINGLE_VARIANTS[0]], reference_sources)
    for variant in SINGLE_VARIANTS[1:]:
        if comparison_inputs(found[variant], reference_sources) != reference_inputs:
            raise ValueError("Single-encoder references have different recorded inputs or splits.")

    # This explicit order preserves the existing tie rule.
    ordered = [{"data": found[variant]} for variant in SINGLE_VARIANTS]
    best, _ = swap_results.validation_choices(
        ordered, singles=SINGLE_VARIANTS, positive_lambdas=(),
    )
    return best


In [ ]:
runs, reference_sources, coverage_rows, reader_issues = {}, {}, [], []
for benchmark in BENCHMARKS:
    found, issues, sources = read_design_results(benchmark)
    reference_sources[benchmark] = sources
    missing = [
        variant for variant in ("concat", "cross", *SINGLE_VARIANTS, *DESIGNS)
        if variant not in found
    ]
    selection_issue = ""
    try:
        best = validation_selected_single(found, sources)
    except ValueError as error:
        best = None
        selection_issue = str(error)
    found["best single"] = found[best] if best is not None else None
    found["best single name"] = best
    runs[benchmark] = found
    coverage_rows.append({
        "benchmark": benchmark,
        "missing or excluded fits": ", ".join(missing),
        "best single": best or "unavailable",
        "selection issue": selection_issue,
        "input provenance": (
            comparison_provenance(*(found[variant] for variant in SINGLE_VARIANTS))
            if best is not None else "unavailable"
        ),
    })
    reader_issues.extend({"benchmark": benchmark, **issue} for issue in issues)

display(pd.DataFrame(coverage_rows).set_index("benchmark"))
if reader_issues:
    display(pd.DataFrame(reader_issues))


,missing or excluded fits,best single,selection issue,input provenance
benchmark,,,,
ecoli_proteins,"mamba_bottleneck, coupled_mamba, uncoupled_mamba",unavailable,Legacy reference reuse: file checksum differs:...,unavailable
cov_vac_degradation,"mamba_bottleneck, coupled_mamba, uncoupled_mamba",unavailable,Legacy reference reuse: file checksum differs:...,unavailable
fungal_expression,"mamba_bottleneck, coupled_mamba, uncoupled_mamba",unavailable,Legacy reference reuse: file checksum differs:...,unavailable
mrna_stability,"mamba_bottleneck, coupled_mamba, uncoupled_mamba",unavailable,Legacy reference reuse: file checksum differs:...,unavailable


,benchmark,source,variant,reason
0,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,concat,configuration differs: seed
1,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,concat,configuration differs: seed
2,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,cross,configuration differs: seed
3,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,cross,configuration differs: seed
4,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,unimodal:prot,configuration differs: seed
5,ecoli_proteins,/Users/julianmckinley/Desktop/Masters/Capstone...,unimodal:prot,configuration differs: seed
6,cov_vac_degradation,/Users/julianmckinley/Desktop/Masters/Capstone...,concat,configuration differs: seed
7,cov_vac_degradation,/Users/julianmckinley/Desktop/Masters/Capstone...,concat,configuration differs: seed
8,cov_vac_degradation,/Users/julianmckinley/Desktop/Masters/Capstone...,cross,configuration differs: seed
9,cov_vac_degradation,/Users/julianmckinley/Desktop/Masters/Capstone...,cross,configuration differs: seed


In [ ]:
scale = {
    benchmark: 100.0 if benchmark == "ecoli_proteins" else 1.0
    for benchmark in BENCHMARKS
}
table = pd.DataFrame({
    benchmark: {
        variant: (
            record["test_score"] * scale[benchmark] if record is not None else np.nan
        )
        for variant, record in runs[benchmark].items()
        if variant != "best single name"
    }
    for benchmark in BENCHMARKS
}).round(3)
display(table)


,ecoli_proteins,cov_vac_degradation,fungal_expression,mrna_stability
concat,60.674,0.801,0.793,0.558
cross,53.792,0.832,0.824,0.553
unimodal:dna,47.753,0.779,0.813,0.539
unimodal-grid:rna,49.298,0.823,0.794,0.547
unimodal:prot,50.140,0.815,0.706,0.531
unimodal:rna,50.702,0.833,0.722,0.552
best single,NaN,NaN,NaN,NaN


## The contrasts that answer the question

With `RUN_BOOTSTRAP = True`, each eligible row is a paired bootstrap on the same test rows under the declared evidence policy: the first run minus the second, with a 95% interval. The check compares ordered-input, target and split fingerprints before resampling and verifies each saved target array against that evidence. The input-provenance column distinguishes original recorded inputs from qualified historical reference reuse; an interval does not remove that historical uncertainty. Missing or mismatched evidence produces an explanatory note, not a numerical contrast. **Resolved** means the interval excludes zero. Read them in this order:

1. **Bottleneck minus concat.** Does squeezing each modality to width 128 cost anything? If so, every Mamba design starts behind and the later contrasts should be read with that in mind.
2. **Uncoupled minus bottleneck.** Does carrying state along the codons help, before any coupling?
3. **Coupled minus uncoupled.** Does coupling the modalities' states help, beyond the recurrence? This is the contrast the design exists for.
4. **Coupled minus concat** and **coupled minus cross.** Where does Coupled Mamba stand against the heads we already have?
5. **Coupled minus best single.** The project's standing question: does fusion beat the best single encoder at all?

In [ ]:
CONTRASTS = (
    ("mamba_bottleneck", "concat"),
    ("uncoupled_mamba", "mamba_bottleneck"),
    ("coupled_mamba", "uncoupled_mamba"),
    ("coupled_mamba", "concat"),
    ("coupled_mamba", "cross"),
    ("coupled_mamba", "best single"),
)


In [ ]:
def paired_design_comparison(first, second, task, reference_sources=None):
    """Require matching evidence under the declared policy before any resampling."""
    first_inputs = comparison_inputs(first, reference_sources)
    second_inputs = comparison_inputs(second, reference_sources)
    if first_inputs != second_inputs:
        raise ValueError("Recorded ordered inputs, targets or splits differ.")
    if first_inputs["targets"]["task"] != task:
        raise ValueError("Recorded target task differs from this benchmark.")

    for record in (first, second):
        issue = swap_results.prediction_issue(
            record, task=task, n_classes=3 if task == "classification" else None,
        )
        if issue:
            raise ValueError(issue)
        expected = first_inputs["split_targets"]["test"]
        actual = run_identity.array_record(record["test_truth"], dtype=expected["dtype"])
        if actual != expected:
            raise ValueError("Saved test targets do not match their recorded fingerprint.")

    # The existing bootstrap consumes only these two saved arrays per model.
    predictions = [
        SimpleNamespace(
            test_truth=record["test_truth"], test_prediction=record["test_prediction"],
        )
        for record in (first, second)
    ]
    return train.paired_bootstrap(*predictions, task, draws=2000, seed=0)


In [ ]:
if not RUN_BOOTSTRAP:
    rows = []
    print(
        "Paired-contrast report unavailable: no standalone bootstrap report is saved. "
        "Historical cell outputs remain readable; set RUN_BOOTSTRAP = True only for "
        "an explicitly selected new analysis of saved predictions."
    )
else:
    rows = []
    for benchmark in BENCHMARKS:
        found = runs[benchmark]
        task = "classification" if benchmark == "ecoli_proteins" else "regression"
        for first, second in CONTRASTS:
            first_result, second_result = found.get(first), found.get(second)
            if first_result is None or second_result is None:
                rows.append({
                    "benchmark": benchmark, "contrast": f"{first} - {second}", "note": "missing",
                })
                continue
            try:
                comparison = paired_design_comparison(
                    first_result, second_result, task, reference_sources[benchmark],
                )
            except ValueError as error:
                rows.append({
                    "benchmark": benchmark, "contrast": f"{first} - {second}", "note": str(error),
                })
                continue
            score_scale = scale[benchmark]
            rows.append({
                "benchmark": benchmark,
                "contrast": f"{first} - {second}",
                "difference": round(comparison["difference"] * score_scale, 3),
                "ci_low": round(comparison["ci_low"] * score_scale, 3),
                "ci_high": round(comparison["ci_high"] * score_scale, 3),
                "resolved": bool(comparison["ci_low"] > 0 or comparison["ci_high"] < 0),
                "note": "",
                "input provenance": comparison_provenance(first_result, second_result),
            })
    display(pd.DataFrame(rows))

Paired-contrast report unavailable: no standalone bootstrap report is saved. Historical cell outputs remain readable; set RUN_BOOTSTRAP = True only for an explicitly selected new analysis of saved predictions.
